# Notebook 02a: Schema & Weak Supervision

**Project:** Hotel Curator -- Multi-Attribute Hotel Quality Classifier

**Inputs (from NB01):**
- `data/processed/reviews_clean.parquet` -- 20,475 cleaned reviews
- `data/processed/eda_summary.json` -- corpus diagnostic

**Objective:** Generate noisy multi-label "silver" labels for each
review across 8 attributes x 2 polarities = 16 binary labels, using
regex labeling functions (LFs) and an optional zero-shot NLI labeler.
Output a stratified hotel-level split, an abstention pool for NB02b
LLM relabeling, and a self-annotation template for NB04 evaluation.

---

## Pipeline

0. Setup -- imports, config, seed, load NB01 artifacts
1. Schema definition (Pydantic + Enums)
2. Regex labeling functions with negation handling
3. Apply LFs to corpus -- build LF matrix
4. Aggregate to silver labels (binary OR with conflict detection)
5. Zero-shot NLI labeler (optional, GPU-accelerated)
6. Hotel-level stratified split
7. Abstention pool for NB02b
8. Gold-set sample template for self-annotation
9. Persist all artifacts + audit

## Attribute taxonomy (8 attributes)

| Attribute | Description | LF style |
|---|---|---|
| `cleanliness` | Room/facility hygiene | sentiment-driven |
| `room` | Comfort, condition, size | sentiment-driven + extractor for size |
| `staff` | Service quality | sentiment-driven |
| `location` | Accessibility, neighborhood | sentiment-driven |
| `noise` | Quietness | sentiment-driven, negation-prone |
| `wifi` | Internet quality | niche vocab |
| `value` | Price perception | sentiment-driven |
| `breakfast` | Food/F&B | sentiment-driven |

Each attribute has positive and negative polarity, giving 16 binary
heads for the multi-label classifier in NB03.

## Outputs

| Artifact | Description |
|---|---|
| `data/processed/silver_labels.parquet` | 20K x 16 binary labels |
| `data/processed/lf_matrix.parquet` | LF firings for audit |
| `data/processed/abstention_pool.parquet` | Reviews where no LF fired |
| `data/splits/hotel_split_mapping.json` | hotel_id -> {train,val,test} |
| `data/processed/attribute_taxonomy.json` | Schema export |
| `data/audit/lf_audit.json` | Coverage, conflict, prevalence stats |
| `data/audit/gold_set_template.csv` | Stratified sample for self-annotation |


In [1]:
# --- 0.0 Install dependencies (run once per environment) ---
# Already done if you followed the env setup. Uncomment if needed.

# %pip install pandas numpy pyarrow pydantic transformers torch tqdm


In [2]:
# --- 0.1 Imports ---

import json
import random
import re
import warnings
from dataclasses import dataclass
from enum import Enum
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


In [3]:
# --- 0.2 Configuration ---

SEED = 42

# Paths
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
AUDIT_DIR = DATA_DIR / "audit"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWS_PARQUET = PROCESSED_DIR / "reviews_clean.parquet"
EDA_SUMMARY_PATH = PROCESSED_DIR / "eda_summary.json"

# Negation handling for keyword LFs
# Tokens fall into three groups based on corpus EDA:
# (a) standard negators present in any English text
# (b) apostrophe-less contractions (188 'wasnt' / 457 'didnt' / 562
#     'dont' etc. — Kaggle preprocessing was inconsistent so both forms
#     coexist in the corpus)
# (c) one soft negator ('hardly') that proved reliable in the
#     "hardly worth" pattern; other soft negators (barely, lacked,
#     looked, far, nothing) measured as more noisy than useful
NEGATION_TOKENS: frozenset[str] = frozenset({
    # (a) standard
    "not", "no", "never", "nothing", "none", "nor",
    "n't", "without", "neither", "non",
    "didn't", "wasn't", "isn't", "aren't", "weren't",
    "hasn't", "haven't", "hadn't", "couldn't", "wouldn't",
    "shouldn't", "won't", "can't",
    # (b) apostrophe-less contractions
    "wasnt", "didnt", "isnt", "arent", "werent",
    "hasnt", "havent", "hadnt", "couldnt", "wouldnt",
    "shouldnt", "wont", "cant", "dont", "doesnt",
    # (c) measured-reliable soft negator
    "hardly", "scarcely",
})
NEGATION_WINDOW_WORDS = 4

# Room size extraction
SQFT_PER_M2 = 10.7639
ROOM_SIZE_LARGE_THRESHOLD_SQFT = 250.0  # spec value

# Hotel-level split
SPLIT_FRACTIONS = (0.80, 0.10, 0.10)  # train, val, test

# Zero-shot NLI (optional)
RUN_NLI = False  # flip to True to run; ~3-5 min on RTX 5090
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
NLI_SAMPLE_SIZE = 5_000     # how many reviews to score with NLI
NLI_THRESHOLD = 0.70        # entailment score threshold
NLI_BATCH_SIZE = 32

# Gold-set template
GOLD_SET_SIZE = 200

print("Config loaded.")
print(f"  REVIEWS_PARQUET = {REVIEWS_PARQUET}")
print(f"  RUN_NLI         = {RUN_NLI}")


Config loaded.
  REVIEWS_PARQUET = data\processed\reviews_clean.parquet
  RUN_NLI         = False


In [4]:
# --- 0.3 Reproducibility ---


def set_seed(seed: int) -> None:
    """Set random seeds across libraries."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)
print(f"Seed set: {SEED}")


Seed set: 42


In [5]:
# --- 0.4 Load NB01 artifacts ---


def load_nb01_artifacts(
    reviews_path: Path,
    summary_path: Path,
) -> tuple[pd.DataFrame, dict[str, object]]:
    """Load the cleaned reviews and EDA summary from NB01.

    Raises
    ------
    FileNotFoundError
        If either artifact is missing -- run NB01 first.
    """
    if not reviews_path.exists():
        raise FileNotFoundError(
            f"{reviews_path} missing. Run NB01 first."
        )
    if not summary_path.exists():
        raise FileNotFoundError(
            f"{summary_path} missing. Run NB01 first."
        )
    df = pd.read_parquet(reviews_path)
    with summary_path.open() as f:
        summary = json.load(f)
    return df, summary


df_reviews, eda_summary = load_nb01_artifacts(
    REVIEWS_PARQUET, EDA_SUMMARY_PATH
)

# Sanity check the schema NB01 promised
expected_cols = ["review_id", "hotel_id", "text", "rating"]
assert list(df_reviews.columns) == expected_cols, (
    f"NB01 schema mismatch: {df_reviews.columns.tolist()}"
)
assert df_reviews["review_id"].is_unique
assert df_reviews["text"].notna().all()

print(f"Loaded {len(df_reviews):,} reviews from NB01")
print(f"Hotels: {df_reviews['hotel_id'].nunique():,}")
print(f"Granularity (per NB01): {eda_summary.get('granularity')}")


Loaded 20,475 reviews from NB01
Hotels: 200
Granularity (per NB01): document


---

## Section 1: Schema Definition

Pydantic models give us a single source of truth for the label
taxonomy. The schema is exported to JSON for NB02b (LLM relabeling)
and NB03 (classifier head naming) to read directly.


In [6]:
# --- 1.1 Attribute and Polarity enums ---


class Attribute(str, Enum):
    """The 8 hotel-quality attributes the classifier predicts."""

    CLEANLINESS = "cleanliness"
    ROOM = "room"
    STAFF = "staff"
    LOCATION = "location"
    NOISE = "noise"
    WIFI = "wifi"
    VALUE = "value"
    BREAKFAST = "breakfast"


class Polarity(str, Enum):
    """Each attribute is detected at positive or negative polarity."""

    POSITIVE = "positive"
    NEGATIVE = "negative"


def label_name(attr: Attribute, pol: Polarity) -> str:
    """Canonical column name for an (attribute, polarity) pair."""
    return f"{attr.value}_{pol.value}"


ALL_LABELS: list[str] = [
    label_name(a, p) for a in Attribute for p in Polarity
]
print(f"Total binary labels: {len(ALL_LABELS)}")
print(f"Labels: {ALL_LABELS}")


Total binary labels: 16
Labels: ['cleanliness_positive', 'cleanliness_negative', 'room_positive', 'room_negative', 'staff_positive', 'staff_negative', 'location_positive', 'location_negative', 'noise_positive', 'noise_negative', 'wifi_positive', 'wifi_negative', 'value_positive', 'value_negative', 'breakfast_positive', 'breakfast_negative']


In [7]:
# --- 1.2 Pydantic schema and export ---


class LabelVote(BaseModel):
    """A single LF or NLI vote on one (attribute, polarity)."""

    attribute: Attribute
    polarity: Polarity
    source: str = Field(
        ..., description="LF name or 'nli' or 'llm' or 'gold'"
    )
    fired: bool


class ReviewLabel(BaseModel):
    """Aggregated multi-label record for one review."""

    review_id: int
    hotel_id: int
    labels: dict[str, bool] = Field(
        ...,
        description="Maps `<attribute>_<polarity>` to bool.",
    )
    abstained: bool = Field(
        ...,
        description="True if no LF fired on this review.",
    )


# Export the schema to disk for downstream notebooks
TAXONOMY_PATH = PROCESSED_DIR / "attribute_taxonomy.json"
taxonomy: dict[str, object] = {
    "attributes": [a.value for a in Attribute],
    "polarities": [p.value for p in Polarity],
    "labels": ALL_LABELS,
    "review_label_schema": ReviewLabel.model_json_schema(),
    "label_vote_schema": LabelVote.model_json_schema(),
}
with TAXONOMY_PATH.open("w") as f:
    json.dump(taxonomy, f, indent=2)
print(f"Exported taxonomy: {TAXONOMY_PATH}")


Exported taxonomy: data\processed\attribute_taxonomy.json


---

## Section 2: Regex Labeling Functions

Each LF is a pure function `text -> bool`. LFs come in two shapes:
- **Phrase LFs**: substring match, no negation guard (phrases are
  already specific enough -- e.g., `"great location"` is unambiguous).
- **Keyword LFs**: word-boundary match with a 4-word lookback for
  negation tokens. Skips firing if `not`, `no`, `never`, `n't`, etc.
  appear within the window before the keyword.

LFs are designed for **lowercase, partially-punctuated** text, which
is what NB01's diagnostic confirmed for this corpus. There is no
`re.IGNORECASE` -- we know everything is lowercase -- and no reliance
on terminal punctuation.

### Why not Snorkel's LabelModel?

Snorkel shines when there are 5+ noisy LFs per task with conflicting
votes that need a learned aggregation. Our 16 LF objects each cover
a distinct (attribute, polarity) cell with mostly-disjoint signals.
Majority vote (here: any-LF-fires) is equivalent to Snorkel's output
in this regime. We can layer Snorkel on later if we add many
overlapping LFs.


In [8]:
# --- 2.1 KeywordLF dataclass and helpers ---


@dataclass(frozen=True)
class KeywordLF:
    """Specification for a keyword/phrase labeling function.

    A KeywordLF fires if any of `phrases` appears as a substring
    without preceding negation in `negation_window` words, OR if
    any of `single_keywords` appears as a whole word without
    preceding negation.
    """

    name: str
    attribute: Attribute
    polarity: Polarity
    single_keywords: tuple[str, ...] = ()
    phrases: tuple[str, ...] = ()
    negation_window: int = NEGATION_WINDOW_WORDS


def _has_negation_before(
    text: str,
    char_pos: int,
    n_words: int,
) -> bool:
    """True if a negation token appears within `n_words` before
    `char_pos`."""
    prefix = text[:char_pos]
    words = prefix.split()
    return any(
        w.strip(",;.:") in NEGATION_TOKENS for w in words[-n_words:]
    )


def keyword_lf_fires(text: str, lf: KeywordLF) -> bool:
    """Apply a single KeywordLF to a single review.

    The negation guard applies to BOTH phrases and single keywords:
    'not great location' does not fire location_pos.

    Negative-cell phrases that self-contain a negation word (e.g.,
    'not clean', 'no service', 'not worth') still fire correctly
    because the guard inspects tokens BEFORE the phrase start, not
    within it. Only rare double-negation patterns like 'not not
    clean' are spuriously suppressed; this is an acceptable trade
    given the corpus.
    """
    # Phrase pass -- substring match WITH negation guard
    for phrase in lf.phrases:
        for m in re.finditer(re.escape(phrase), text):
            if not _has_negation_before(
                text, m.start(), lf.negation_window
            ):
                return True
    # Keyword pass -- word boundary + negation guard
    for kw in lf.single_keywords:
        for m in re.finditer(rf"\b{re.escape(kw)}\b", text):
            if not _has_negation_before(
                text, m.start(), lf.negation_window
            ):
                return True
    return False


In [9]:
# --- 2.2 LF library (16 LFs covering 8 attributes x 2 polarities) ---

LF_REGISTRY: list[KeywordLF] = [
    # ---- CLEANLINESS ----
    KeywordLF(
        name="cleanliness_pos",
        attribute=Attribute.CLEANLINESS,
        polarity=Polarity.POSITIVE,
        single_keywords=(
            "clean", "spotless", "immaculate", "tidy",
            "pristine", "sanitary",
        ),
        phrases=(
            "well kept", "well maintained", "very clean",
            "extremely clean", "super clean", "spotlessly clean",
        ),
    ),
    KeywordLF(
        name="cleanliness_neg",
        attribute=Attribute.CLEANLINESS,
        polarity=Polarity.NEGATIVE,
        single_keywords=(
            "dirty", "filthy", "grimy", "smelly",
            "stained", "gross", "disgusting", "moldy", "mouldy",
        ),
        phrases=(
            "not clean", "needs cleaning", "needed cleaning",
            "dust everywhere",
        ),
    ),
    # ---- ROOM ----
    KeywordLF(
        name="room_pos",
        attribute=Attribute.ROOM,
        polarity=Polarity.POSITIVE,
        single_keywords=(
            "spacious", "comfortable", "cozy", "modern", "updated",
            "renovated", "stylish",
        ),
        phrases=(
            "large room", "big room", "huge room", "nice room",
            "great room", "lovely room", "comfy bed", "comfortable bed",
            "king bed",
        ),
    ),
    KeywordLF(
        name="room_neg",
        attribute=Attribute.ROOM,
        polarity=Polarity.NEGATIVE,
        single_keywords=(
            "cramped", "tiny", "outdated", "shabby", "dingy",
            "dated", "rundown", "broken",
        ),
        phrases=(
            "small room", "small rooms", "tiny room", "old room",
            "uncomfortable bed", "hard bed", "thin walls",
        ),
    ),
    # ---- STAFF ----
    KeywordLF(
        name="staff_pos",
        attribute=Attribute.STAFF,
        polarity=Polarity.POSITIVE,
        single_keywords=(
            "friendly", "helpful", "attentive", "professional",
            "courteous", "accommodating", "welcoming",
        ),
        phrases=(
            "great staff", "great service", "excellent staff",
            "excellent service", "wonderful staff", "amazing staff",
            "great concierge", "helpful staff", "polite staff",
        ),
    ),
    KeywordLF(
        name="staff_neg",
        attribute=Attribute.STAFF,
        polarity=Polarity.NEGATIVE,
        single_keywords=(
            "rude", "unhelpful", "unfriendly", "indifferent",
            "unprofessional",
        ),
        phrases=(
            "bad service", "poor service", "terrible service",
            "rude staff", "slow service", "no service",
        ),
    ),
    # ---- LOCATION ----
    KeywordLF(
        name="location_pos",
        attribute=Attribute.LOCATION,
        polarity=Polarity.POSITIVE,
        single_keywords=("central", "convenient",),
        phrases=(
            "great location", "perfect location", "good location",
            "excellent location", "ideal location", "amazing location",
            "close to", "near to", "walking distance",
            "short walk", "easy walk",
        ),
    ),
    KeywordLF(
        name="location_neg",
        attribute=Attribute.LOCATION,
        polarity=Polarity.NEGATIVE,
        single_keywords=("remote", "isolated", "inconvenient",),
        phrases=(
            "bad location", "poor location", "far from",
            "hard to find", "middle of nowhere", "sketchy area",
            "dodgy area",
        ),
    ),
    # ---- NOISE ----
    KeywordLF(
        name="noise_pos",
        attribute=Attribute.NOISE,
        polarity=Polarity.POSITIVE,
        single_keywords=("quiet", "peaceful", "tranquil", "silent",),
        phrases=("very quiet", "extremely quiet", "no noise",),
    ),
    KeywordLF(
        name="noise_neg",
        attribute=Attribute.NOISE,
        polarity=Polarity.NEGATIVE,
        single_keywords=("noisy", "loud",),
        phrases=(
            "thin walls", "could hear", "traffic noise",
            "street noise", "construction noise", "very noisy",
            "kept us awake", "couldn't sleep", "couldnt sleep",
        ),
    ),
    # ---- WIFI ----
    KeywordLF(
        name="wifi_pos",
        attribute=Attribute.WIFI,
        polarity=Polarity.POSITIVE,
        phrases=(
            "good wifi", "fast wifi", "free wifi", "great wifi",
            "reliable wifi", "strong wifi",
            "good internet", "fast internet", "free internet",
        ),
    ),
    KeywordLF(
        name="wifi_neg",
        attribute=Attribute.WIFI,
        polarity=Polarity.NEGATIVE,
        phrases=(
            "slow wifi", "bad wifi", "no wifi", "poor wifi",
            "weak wifi", "expensive wifi",
            "wifi was slow", "wifi was bad", "wifi did n",
            "no internet", "bad internet", "slow internet",
            "internet was slow",
        ),
    ),
    # ---- VALUE ----
    KeywordLF(
        name="value_pos",
        attribute=Attribute.VALUE,
        polarity=Polarity.POSITIVE,
        phrases=(
            "good value", "great value", "worth it", "great deal",
            "good deal", "value for money", "good price",
            "reasonable price", "reasonably priced",
        ),
    ),
    KeywordLF(
        name="value_neg",
        attribute=Attribute.VALUE,
        polarity=Polarity.NEGATIVE,
        single_keywords=("overpriced",),
        phrases=(
            "not worth", "rip off", "rip-off", "expensive parking",
            "extra charge", "hidden fees", "overpriced",
        ),
    ),
    # ---- BREAKFAST ----
    KeywordLF(
        name="breakfast_pos",
        attribute=Attribute.BREAKFAST,
        polarity=Polarity.POSITIVE,
        phrases=(
            "great breakfast", "good breakfast", "free breakfast",
            "delicious breakfast", "amazing breakfast",
            "nice breakfast", "wonderful breakfast",
            "breakfast was great", "breakfast was good",
        ),
    ),
    KeywordLF(
        name="breakfast_neg",
        attribute=Attribute.BREAKFAST,
        polarity=Polarity.NEGATIVE,
        phrases=(
            "bad breakfast", "no breakfast", "poor breakfast",
            "cold breakfast", "expensive breakfast", "limited breakfast",
            "breakfast was bad", "breakfast was poor",
        ),
    ),
]

print(f"LF library: {len(LF_REGISTRY)} LFs")
print(f"Coverage: {len({lf.attribute for lf in LF_REGISTRY})} attributes "
      f"x {len({lf.polarity for lf in LF_REGISTRY})} polarities")


LF library: 16 LFs
Coverage: 8 attributes x 2 polarities


In [10]:
# --- 2.3 Room size extractor + LF self-test ---

ROOM_SIZE_SQFT_RE = re.compile(
    r"\b(\d{2,4})\s*(?:sq\s?ft|square\s?fe?e?t|square\s?foot)\b"
)
ROOM_SIZE_M2_RE = re.compile(
    r"\b(\d{1,3})\s*(?:m2|sq\s?m|square\s?meters?)\b"
)


def extract_room_size_sqft(text: str) -> float | None:
    """Extract room size in square feet from text, or None."""
    if m := ROOM_SIZE_SQFT_RE.search(text):
        return float(m.group(1))
    if m := ROOM_SIZE_M2_RE.search(text):
        return float(m.group(1)) * SQFT_PER_M2
    return None


def lf_room_size_large(text: str) -> bool:
    """Fire if extracted room size meets the 'large' threshold."""
    size = extract_room_size_sqft(text)
    return size is not None and size >= ROOM_SIZE_LARGE_THRESHOLD_SQFT


# Self-test: each LF should fire on its hand-built positive case.
SELF_TEST_CASES: dict[str, str] = {
    "cleanliness_pos": "the room was spotless and clean throughout",
    "cleanliness_neg": "bathroom was filthy and dirty everywhere",
    "room_pos":        "spacious room with a comfortable king bed",
    "room_neg":        "tiny cramped room and uncomfortable bed",
    "staff_pos":       "great staff, very friendly and helpful",
    "staff_neg":       "rude staff and bad service throughout",
    "location_pos":    "great location, walking distance to everything",
    "location_neg":    "bad location, far from city center",
    "noise_pos":       "very quiet and peaceful at night",
    "noise_neg":       "noisy with thin walls, could hear neighbors",
    "wifi_pos":        "free wifi was fast and reliable",
    "wifi_neg":        "slow wifi and no internet in room",
    "value_pos":       "great value for money, good deal",
    "value_neg":       "overpriced and not worth the price",
    "breakfast_pos":   "great breakfast, delicious and varied",
    "breakfast_neg":   "no breakfast included and very expensive",
}

failures = 0
for lf in LF_REGISTRY:
    test_text = SELF_TEST_CASES.get(lf.name, "")
    if not test_text:
        print(f"  WARN: no self-test case for {lf.name}")
        continue
    if not keyword_lf_fires(test_text, lf):
        failures += 1
        print(f"  FAIL: {lf.name} did not fire on '{test_text}'")

# Negation guard tests
# (a) Single-word keyword negation
neg_text_kw = "the room was not clean and not spotless either"
lf_clean_pos = next(
    lf for lf in LF_REGISTRY if lf.name == "cleanliness_pos"
)
if keyword_lf_fires(neg_text_kw, lf_clean_pos):
    failures += 1
    print("  FAIL: cleanliness_pos fired on 'not clean'")

# (b) Phrase negation -- the bug this fix targets
neg_text_phrase = "not great location and not good value at all"
lf_loc_pos = next(
    lf for lf in LF_REGISTRY if lf.name == "location_pos"
)
lf_val_pos = next(
    lf for lf in LF_REGISTRY if lf.name == "value_pos"
)
if keyword_lf_fires(neg_text_phrase, lf_loc_pos):
    failures += 1
    print("  FAIL: location_pos fired on 'not great location'")
if keyword_lf_fires(neg_text_phrase, lf_val_pos):
    failures += 1
    print("  FAIL: value_pos fired on 'not good value'")

# (c) Apostrophe-less contraction negation
neg_text_apos = "the room wasnt clean and didnt have hot water"
if keyword_lf_fires(neg_text_apos, lf_clean_pos):
    failures += 1
    print("  FAIL: cleanliness_pos fired on 'wasnt clean'")

# (d) Self-negating phrases in NEGATIVE cells must still fire
self_neg_text = "the bathroom was not clean at all"
lf_clean_neg = next(
    lf for lf in LF_REGISTRY if lf.name == "cleanliness_neg"
)
if not keyword_lf_fires(self_neg_text, lf_clean_neg):
    failures += 1
    print("  FAIL: cleanliness_neg did not fire on 'not clean'")

if failures == 0:
    print("LF self-test: PASS (16/16 LFs + 5 negation guard tests)")
else:
    raise AssertionError(f"{failures} LF self-tests failed")

LF self-test: PASS (16/16 LFs + 5 negation guard tests)


---

## Section 3: Apply LFs to Corpus

We apply each of the 16 LFs to all 20K reviews and build an LF
matrix (`reviews x LFs`, boolean). Vectorized phrase matching is
done with `str.contains`; the negation-guarded keyword pass falls
back to a Python loop on the small subset where keywords match
(typically <10% of rows per LF).


In [11]:
# --- 3.1 Apply LFs to corpus (vectorized) ---


def apply_keyword_lf_vectorized(
    series: pd.Series,
    lf: KeywordLF,
) -> pd.Series:
    """Vectorized LF application returning a bool Series.

    Phrases are checked via str.contains (no negation guard).
    Keywords are first checked with a vectorized regex; rows that
    match are then re-checked with the per-row negation guard.
    """
    fired = pd.Series(False, index=series.index)

    # Phrase pass
    if lf.phrases:
        pattern = "|".join(re.escape(p) for p in lf.phrases)
        fired |= series.str.contains(pattern, regex=True, na=False)

    # Keyword pass: vectorized prefilter, then per-row negation guard
    if lf.single_keywords:
        kw_pattern = (
            r"\b(?:" + "|".join(
                re.escape(k) for k in lf.single_keywords
            ) + r")\b"
        )
        candidate_mask = series.str.contains(
            kw_pattern, regex=True, na=False
        ) & ~fired
        candidate_idx = candidate_mask[candidate_mask].index
        for idx in candidate_idx:
            text = series.loc[idx]
            if keyword_lf_fires(text, lf):
                fired.loc[idx] = True

    return fired


# Build the LF matrix: rows = reviews, cols = LF names
print("Applying LFs ...")
lf_matrix = pd.DataFrame(
    index=df_reviews["review_id"],
    columns=[lf.name for lf in LF_REGISTRY],
    dtype=bool,
)
for lf in tqdm(LF_REGISTRY, desc="LFs"):
    lf_matrix[lf.name] = apply_keyword_lf_vectorized(
        df_reviews["text"], lf
    ).values

# Add the room-size extractor as a 17th LF
lf_matrix["room_size_large"] = (
    df_reviews["text"].apply(lf_room_size_large).values
)

print(f"LF matrix: {lf_matrix.shape}")


Applying LFs ...


LFs:   0%|          | 0/16 [00:00<?, ?it/s]

LF matrix: (20475, 17)


In [12]:
# --- 3.2 Per-LF coverage ---

coverage = lf_matrix.sum().sort_values(ascending=False)
total = len(lf_matrix)
print(f"Per-LF coverage (n={total:,}):")
print(f"  {'LF':<22s} {'count':>8s} {'rate':>8s}")
print(f"  {'-' * 22} {'-' * 8} {'-' * 8}")
for lf_name, count in coverage.items():
    rate = count / total * 100
    print(f"  {lf_name:<22s} {count:>8,} {rate:>7.2f}%")

n_with_any = lf_matrix.any(axis=1).sum()
print(
    f"\nReviews with >= 1 LF firing: {n_with_any:,} "
    f"({n_with_any / total * 100:.1f}%)"
)


Per-LF coverage (n=20,475):
  LF                        count     rate
  ---------------------- -------- --------
  staff_pos                 9,719   47.47%
  cleanliness_pos           7,867   38.42%
  room_pos                  7,435   36.31%
  location_pos              6,475   31.62%
  noise_pos                 2,643   12.91%
  value_pos                 2,331   11.38%
  room_neg                  1,951    9.53%
  noise_neg                 1,513    7.39%
  cleanliness_neg           1,117    5.46%
  staff_neg                   915    4.47%
  value_neg                   837    4.09%
  breakfast_pos               692    3.38%
  wifi_pos                    605    2.95%
  location_neg                256    1.25%
  breakfast_neg                94    0.46%
  wifi_neg                     32    0.16%
  room_size_large              25    0.12%

Reviews with >= 1 LF firing: 18,009 (88.0%)


---

## Section 4: Aggregate to Silver Labels

Each LF maps 1:1 to a `(attribute, polarity)` cell in our taxonomy
(plus the room_size_large extractor). Aggregation is straightforward:
the silver label for `<attr>_<pol>` is True iff at least one LF for
that cell fired. We also flag `conflict` -- when both polarities of
the same attribute fire on the same review -- which is useful audit
signal but does not change the binary label.


In [13]:
# --- 4.1 Aggregate LF firings to silver labels ---

# Map each LF to its (attribute, polarity) cell
LF_TO_LABEL: dict[str, str] = {
    lf.name: label_name(lf.attribute, lf.polarity)
    for lf in LF_REGISTRY
}
# room_size_large is a separate signal; map to room_positive
# (large rooms are a positive room signal)
LF_TO_LABEL["room_size_large"] = label_name(
    Attribute.ROOM, Polarity.POSITIVE
)


def aggregate_silver_labels(
    lf_matrix: pd.DataFrame,
    lf_to_label: dict[str, str],
    all_labels: list[str],
) -> pd.DataFrame:
    """OR-aggregate LF firings into one bool column per label."""
    silver = pd.DataFrame(
        False, index=lf_matrix.index, columns=all_labels
    )
    for lf_name, label_col in lf_to_label.items():
        if lf_name in lf_matrix.columns:
            silver[label_col] |= lf_matrix[lf_name]
    return silver


silver_labels = aggregate_silver_labels(
    lf_matrix, LF_TO_LABEL, ALL_LABELS
)

# Conflict detection: both polarities fire for the same attribute
conflicts: dict[str, int] = {}
for attr in Attribute:
    pos_col = label_name(attr, Polarity.POSITIVE)
    neg_col = label_name(attr, Polarity.NEGATIVE)
    n_conflict = int((silver_labels[pos_col] & silver_labels[neg_col]).sum())
    conflicts[attr.value] = n_conflict

print("Per-attribute conflict counts (both polarities fired):")
for attr, n in conflicts.items():
    print(f"  {attr:<14s}: {n:>5,}")


Per-attribute conflict counts (both polarities fired):
  cleanliness   :   338
  room          :   664
  staff         :   288
  location      :    68
  noise         :   269
  wifi          :     6
  value         :   111
  breakfast     :     3


In [14]:
# --- 4.2 Per-attribute prevalence and abstention rate ---

prevalence = silver_labels.sum().sort_values(ascending=False)
print("Per-label prevalence:")
print(f"  {'label':<26s} {'count':>8s} {'rate':>8s}")
print(f"  {'-' * 26} {'-' * 8} {'-' * 8}")
for col, count in prevalence.items():
    rate = count / len(silver_labels) * 100
    print(f"  {col:<26s} {count:>8,} {rate:>7.2f}%")

abstained = ~silver_labels.any(axis=1)
n_abstain = int(abstained.sum())
print(
    f"\nAbstention (no label fired): {n_abstain:,} "
    f"({n_abstain / len(silver_labels) * 100:.1f}%)"
)


Per-label prevalence:
  label                         count     rate
  -------------------------- -------- --------
  staff_positive                9,719   47.47%
  cleanliness_positive          7,867   38.42%
  room_positive                 7,447   36.37%
  location_positive             6,475   31.62%
  noise_positive                2,643   12.91%
  value_positive                2,331   11.38%
  room_negative                 1,951    9.53%
  noise_negative                1,513    7.39%
  cleanliness_negative          1,117    5.46%
  staff_negative                  915    4.47%
  value_negative                  837    4.09%
  breakfast_positive              692    3.38%
  wifi_positive                   605    2.95%
  location_negative               256    1.25%
  breakfast_negative               94    0.46%
  wifi_negative                    32    0.16%

Abstention (no label fired): 2,466 (12.0%)


---

## Section 5: Zero-Shot NLI Labeler (Optional)

Set `RUN_NLI = True` in the config cell to run. We use
`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` (~280 MB) on a sample
of `NLI_SAMPLE_SIZE` reviews. For each review we score 16 hypotheses
(one per `(attribute, polarity)` pair) and treat scores above
`NLI_THRESHOLD` as positive labels.

NLI labels are **OR-merged** with the regex silver labels. This
boosts recall on lexically-implicit cases (e.g., "rooms had a
musty smell" -> CLEANLINESS negative without the keyword "dirty").

Wall time on RTX 5090: ~3-5 min for 5K reviews. Skip on CPU.


In [15]:
# --- 5.1 NLI hypotheses ---

NLI_HYPOTHESES: dict[str, str] = {
    label_name(Attribute.CLEANLINESS, Polarity.POSITIVE):
        "The hotel room was clean.",
    label_name(Attribute.CLEANLINESS, Polarity.NEGATIVE):
        "The hotel room was dirty.",
    label_name(Attribute.ROOM, Polarity.POSITIVE):
        "The room was comfortable and well-appointed.",
    label_name(Attribute.ROOM, Polarity.NEGATIVE):
        "The room was uncomfortable or in poor condition.",
    label_name(Attribute.STAFF, Polarity.POSITIVE):
        "The staff were friendly and helpful.",
    label_name(Attribute.STAFF, Polarity.NEGATIVE):
        "The staff were rude or unhelpful.",
    label_name(Attribute.LOCATION, Polarity.POSITIVE):
        "The hotel was in a good location.",
    label_name(Attribute.LOCATION, Polarity.NEGATIVE):
        "The hotel was in a bad or inconvenient location.",
    label_name(Attribute.NOISE, Polarity.POSITIVE):
        "The hotel was quiet.",
    label_name(Attribute.NOISE, Polarity.NEGATIVE):
        "The hotel was noisy.",
    label_name(Attribute.WIFI, Polarity.POSITIVE):
        "The wifi or internet was fast and reliable.",
    label_name(Attribute.WIFI, Polarity.NEGATIVE):
        "The wifi or internet was slow or unreliable.",
    label_name(Attribute.VALUE, Polarity.POSITIVE):
        "The hotel was good value for money.",
    label_name(Attribute.VALUE, Polarity.NEGATIVE):
        "The hotel was overpriced.",
    label_name(Attribute.BREAKFAST, Polarity.POSITIVE):
        "The breakfast was good.",
    label_name(Attribute.BREAKFAST, Polarity.NEGATIVE):
        "The breakfast was bad or limited.",
}

assert set(NLI_HYPOTHESES.keys()) == set(ALL_LABELS)
print(f"NLI hypotheses defined: {len(NLI_HYPOTHESES)}")


NLI hypotheses defined: 16


In [16]:
# --- 5.2 Run NLI labeler (if RUN_NLI) ---


def build_nli_pipeline(model_name: str):
    """Lazy-import torch/transformers; build a zero-shot pipeline."""
    import torch
    from transformers import pipeline
    device = 0 if torch.cuda.is_available() else -1
    print(f"NLI device: {'cuda:0' if device == 0 else 'cpu'}")
    return pipeline(
        "zero-shot-classification",
        model=model_name,
        device=device,
    )


def score_nli_batch(
    classifier,
    texts: list[str],
    hypotheses_by_label: dict[str, str],
    batch_size: int,
    threshold: float,
) -> pd.DataFrame:
    """Score each text against all hypotheses; return bool DataFrame."""
    labels_in_order = list(hypotheses_by_label.keys())
    candidate_labels = [hypotheses_by_label[k] for k in labels_in_order]

    rows: list[dict[str, bool]] = []
    for i in tqdm(
        range(0, len(texts), batch_size), desc="NLI"
    ):
        batch = texts[i:i + batch_size]
        results = classifier(
            batch, candidate_labels=candidate_labels,
            multi_label=True,
        )
        if isinstance(results, dict):
            results = [results]
        for r in results:
            score_by_label: dict[str, float] = dict(
                zip(r["labels"], r["scores"], strict=True)
            )
            row = {
                lbl: score_by_label.get(
                    hypotheses_by_label[lbl], 0.0
                ) >= threshold
                for lbl in labels_in_order
            }
            rows.append(row)

    return pd.DataFrame(rows, columns=labels_in_order)


nli_labels = pd.DataFrame(
    False, index=df_reviews["review_id"], columns=ALL_LABELS
)

if RUN_NLI:
    sample_size = min(NLI_SAMPLE_SIZE, len(df_reviews))
    sample_df = df_reviews.sample(
        sample_size, random_state=SEED
    ).reset_index(drop=True)

    classifier = build_nli_pipeline(NLI_MODEL)
    nli_sample = score_nli_batch(
        classifier,
        sample_df["text"].tolist(),
        NLI_HYPOTHESES,
        batch_size=NLI_BATCH_SIZE,
        threshold=NLI_THRESHOLD,
    )
    nli_sample.index = sample_df["review_id"].values
    # Place sampled NLI labels into the full-corpus frame
    nli_labels.loc[nli_sample.index] = nli_sample.values
    print(f"NLI labels populated for {len(nli_sample):,} reviews")
else:
    print("RUN_NLI = False; skipping NLI pass (using regex labels only)")


RUN_NLI = False; skipping NLI pass (using regex labels only)


In [17]:
# --- 5.3 Merge NLI labels into silver (OR-aggregation) ---

silver_pre_nli = silver_labels.copy()
nli_aligned = nli_labels.reindex(
    silver_labels.index, fill_value=False
)
silver_labels = silver_labels | nli_aligned.values

if RUN_NLI:
    delta = (silver_labels.sum() - silver_pre_nli.sum())
    print("Per-label uplift from NLI:")
    for col, d in delta.items():
        print(f"  {col:<26s}: +{int(d):,}")
else:
    print("Silver labels unchanged (NLI was off).")


Silver labels unchanged (NLI was off).


---

## Section 6: Hotel-Level Stratified Split

The reviews from the same hotel
**never** cross the train/val/test boundary, so the classifier can't
memorize hotel-specific phrasing. We stratify on rating to keep the
label distribution comparable across splits.

For our 200 synthetic hotels, an 80/10/10 split is straightforward.
The split mapping is persisted as JSON for NB02b and NB03 to consume.


In [18]:
# --- 6.1 Hotel-level stratified split ---


def stratified_hotel_split(
    df: pd.DataFrame,
    *,
    fractions: tuple[float, float, float],
    seed: int,
) -> dict[str, list[int]]:
    """Assign each hotel to train/val/test, stratified on its
    median rating.

    Returns
    -------
    dict[str, list[int]]
        Keys: 'train', 'val', 'test'. Values: hotel_id lists.
    """
    train_frac, val_frac, _ = fractions
    rng = np.random.default_rng(seed)

    # Bucket each hotel by its median rating (1-5 -> 5 buckets)
    hotel_median = (
        df.groupby("hotel_id")["rating"].median().astype(int)
    )

    splits: dict[str, list[int]] = {
        "train": [], "val": [], "test": []
    }
    for bucket in sorted(hotel_median.unique()):
        ids = hotel_median[hotel_median == bucket].index.tolist()
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        splits["train"].extend(ids[:n_train])
        splits["val"].extend(ids[n_train:n_train + n_val])
        splits["test"].extend(ids[n_train + n_val:])

    return splits


hotel_splits = stratified_hotel_split(
    df_reviews,
    fractions=SPLIT_FRACTIONS,
    seed=SEED,
)

for name, ids in hotel_splits.items():
    n_reviews = df_reviews["hotel_id"].isin(ids).sum()
    print(
        f"  {name:5s}: {len(ids):>3} hotels, "
        f"{n_reviews:>6,} reviews"
    )

# Persist the mapping
SPLIT_PATH = SPLITS_DIR / "hotel_split_mapping.json"
with SPLIT_PATH.open("w") as f:
    json.dump(
        {k: [int(x) for x in v] for k, v in hotel_splits.items()},
        f, indent=2,
    )
print(f"Saved: {SPLIT_PATH}")


  train: 160 hotels, 16,412 reviews
  val  :  20 hotels,  2,046 reviews
  test :  20 hotels,  2,017 reviews
Saved: data\splits\hotel_split_mapping.json


---

## Section 7: Abstention Pool for NB02b

Reviews where **no LF fired** become the abstention pool for NB02b's
LLM relabeling pass. To keep the validation/test sets pristine,
the pool is restricted to **train hotels only** -- LLM relabeling
is a training-side augmentation.


In [19]:
# --- 7.1 Build abstention pool (train hotels only) ---

train_hotels = set(hotel_splits["train"])
in_train = df_reviews["hotel_id"].isin(train_hotels)

# Recompute abstention against the post-NLI silver_labels
abstained_mask = ~silver_labels.any(axis=1)
abstained_mask = abstained_mask.reindex(
    df_reviews["review_id"].values
).values

abstention_pool = df_reviews[in_train.values & abstained_mask].copy()
abstention_path = PROCESSED_DIR / "abstention_pool.parquet"
abstention_pool.to_parquet(abstention_path, index=False)

print(f"Abstention pool: {len(abstention_pool):,} reviews")
print(
    f"  ({len(abstention_pool) / in_train.sum() * 100:.1f}% of "
    f"train reviews)"
)
print(f"Saved: {abstention_path}")


Abstention pool: 1,990 reviews
  (12.1% of train reviews)
Saved: data\processed\abstention_pool.parquet


---

## Section 8: Gold-Set Template for Self-Annotation

We export 200 reviews stratified by LF-firing diversity for
self-annotation. The template is a CSV with one column per label
that the annotator (you) marks 0/1/?. The annotated CSV becomes
the gold set NB04 will evaluate against.

Stratification: half from "low-coverage" reviews (1-2 LFs fired)
and half from "high-coverage" reviews (3+ LFs fired). This surfaces
both the easy cases (sanity check) and the hard cases (informative
errors).


In [20]:
# --- 8.1 Gold-set template ---


def build_gold_template(
    df_reviews: pd.DataFrame,
    silver: pd.DataFrame,
    all_labels: list[str],
    *,
    n_total: int,
    seed: int,
) -> pd.DataFrame:
    """Sample n_total reviews stratified by LF-firing density.

    Returns a CSV-ready DataFrame with empty annotation columns.
    """
    silver_aligned = silver.reindex(df_reviews["review_id"].values)
    fires_per_review = silver_aligned.sum(axis=1).values
    n_low = n_total // 2
    n_high = n_total - n_low

    rng = np.random.default_rng(seed)
    low_idx = np.where((fires_per_review >= 1) & (fires_per_review <= 2))[0]
    high_idx = np.where(fires_per_review >= 3)[0]

    low_pick = rng.choice(
        low_idx, size=min(n_low, len(low_idx)), replace=False
    )
    high_pick = rng.choice(
        high_idx, size=min(n_high, len(high_idx)), replace=False
    )

    pick = np.concatenate([low_pick, high_pick])
    rng.shuffle(pick)

    out = df_reviews.iloc[pick][
        ["review_id", "hotel_id", "rating", "text"]
    ].reset_index(drop=True)
    for lbl in all_labels:
        out[lbl] = ""  # annotator fills 0 / 1 / ?
    return out


gold_template = build_gold_template(
    df_reviews, silver_labels, ALL_LABELS,
    n_total=GOLD_SET_SIZE, seed=SEED,
)
gold_path = AUDIT_DIR / "gold_set_template.csv"
gold_template.to_csv(gold_path, index=False)
print(f"Gold-set template ({len(gold_template):,} rows): {gold_path}")
print(
    "Open in Excel/sheets, mark each label column with 0 / 1 / ?, "
    "save as `gold_set_annotated.csv` next to this file. "
    "NB04 will read it for evaluation."
)


Gold-set template (200 rows): data\audit\gold_set_template.csv
Open in Excel/sheets, mark each label column with 0 / 1 / ?, save as `gold_set_annotated.csv` next to this file. NB04 will read it for evaluation.


---

## Section 9: Persist Artifacts and Audit


In [21]:
# --- 9.1 Persist silver labels, LF matrix, audit ---

# Reset index so review_id is a column, not the index
silver_out = silver_labels.reset_index().rename(
    columns={"index": "review_id"}
)
# Ensure review_id is the first column
cols = ["review_id"] + [c for c in silver_out.columns if c != "review_id"]
silver_out = silver_out[cols]

silver_path = PROCESSED_DIR / "silver_labels.parquet"
silver_out.to_parquet(silver_path, index=False)

lf_matrix_out = lf_matrix.reset_index().rename(
    columns={"index": "review_id"}
)
lf_matrix_path = PROCESSED_DIR / "lf_matrix.parquet"
lf_matrix_out.to_parquet(lf_matrix_path, index=False)

# Audit summary
audit: dict[str, object] = {
    "seed": SEED,
    "n_reviews": int(len(df_reviews)),
    "n_lfs": len(LF_REGISTRY) + 1,  # + room_size_large
    "lf_coverage": {
        col: {
            "count": int(lf_matrix[col].sum()),
            "rate": float(lf_matrix[col].mean()),
        }
        for col in lf_matrix.columns
    },
    "label_prevalence": {
        col: {
            "count": int(silver_labels[col].sum()),
            "rate": float(silver_labels[col].mean()),
        }
        for col in silver_labels.columns
    },
    "conflicts_per_attribute": conflicts,
    "abstention_count": int((~silver_labels.any(axis=1)).sum()),
    "abstention_rate": float((~silver_labels.any(axis=1)).mean()),
    "nli_run": RUN_NLI,
    "nli_model": NLI_MODEL if RUN_NLI else None,
    "split_summary": {
        name: {
            "hotels": len(ids),
            "reviews": int(
                df_reviews["hotel_id"].isin(ids).sum()
            ),
        }
        for name, ids in hotel_splits.items()
    },
}
audit_path = AUDIT_DIR / "lf_audit.json"
with audit_path.open("w") as f:
    json.dump(audit, f, indent=2)

print("Saved:")
print(f"  {silver_path}        ({silver_path.stat().st_size / 1e6:.2f} MB)")
print(f"  {lf_matrix_path}     ({lf_matrix_path.stat().st_size / 1e6:.2f} MB)")
print(f"  {abstention_path}    ({abstention_path.stat().st_size / 1e6:.2f} MB)")
print(f"  {SPLIT_PATH}")
print(f"  {gold_path}")
print(f"  {audit_path}")
print(f"  {TAXONOMY_PATH}")


Saved:
  data\processed\silver_labels.parquet        (0.16 MB)
  data\processed\lf_matrix.parquet     (0.16 MB)
  data\processed\abstention_pool.parquet    (0.67 MB)
  data\splits\hotel_split_mapping.json
  data\audit\gold_set_template.csv
  data\audit\lf_audit.json
  data\processed\attribute_taxonomy.json


In [22]:
# --- 9.2 Show example sentences per LF (sanity check) ---

print("Example reviews per LF (first 3 firings):")
print()
for lf_name in [lf.name for lf in LF_REGISTRY[:8]]:  # first 8 LFs
    fired_ids = lf_matrix[lf_matrix[lf_name]].index[:3].tolist()
    if not fired_ids:
        print(f"[{lf_name}] no firings")
        continue
    print(f"[{lf_name}]")
    for rid in fired_ids:
        text = df_reviews.set_index("review_id").loc[rid, "text"]
        snippet = text[:140].replace("\n", " ")
        print(f"  rid={rid}: {snippet}...")
    print()


Example reviews per LF (first 3 firings):

[cleanliness_pos]
  rid=0: nice hotel expensive parking got good deal stay hotel anniversary, arrived late evening took advice previous reviews did valet parking, chec...
  rid=6: cozy stay rainy city, husband spent 7 nights monaco early january 2008. business trip chance come ride.we booked monte carlo suite proved co...
  rid=12: nice hotel good location hotel kimpton design whimsical vibe fun, staff young casual problem hotel busy stay friendly helpful, group reserve...

[cleanliness_neg]
  rid=40: bad choice, booked hotel hot wire called immediately requesting nonsmoking room king bed room away elevator/ice.the person spoke pleasant st...
  rid=42: warwick bad good reviews warwick shocks staff quite rude rooms fairly dirty, cut asked bandaid did not, requested bottle opener did not bett...
  rid=43: great service warwick just recently spent 4 days christmas holidays warwick, helpfulness care courtesy experienced desk restaurant room serv..

---

## Summary

| Artifact | Path | Description |
|---|---|---|
| Silver labels | `data/processed/silver_labels.parquet` | 20K x 16 binary labels for NB03 |
| LF matrix | `data/processed/lf_matrix.parquet` | Per-LF firings for audit |
| Abstention pool | `data/processed/abstention_pool.parquet` | Train-only reviews where no LF fired (NB02b input) |
| Hotel split | `data/splits/hotel_split_mapping.json` | hotel_id -> {train,val,test} |
| Taxonomy | `data/processed/attribute_taxonomy.json` | Schema export |
| LF audit | `data/audit/lf_audit.json` | Coverage, conflict, prevalence stats |
| Gold template | `data/audit/gold_set_template.csv` | 200-row self-annotation sheet |

### Decisions documented

| Decision | Rationale |
|---|---|
| 8 attributes x 2 polarities = 16 binary heads | Spec recommendation; fits BCE multi-label cleanly |
| Majority-vote (any-LF-fires) over Snorkel | Each (attr, pol) cell has only 1-2 LFs; Snorkel's value is in many-LF regimes |
| Negation guard with 4-word lookback | Lowercase corpus has no contraction/cap cues; negation tokens are the only signal |
| Hotel-level stratified split | Prevents review-level leakage; analog of TrialParser's NCT-level split |
| Abstention pool restricted to train hotels | LLM relabeling is a training-side augmentation; preserves val/test purity |
| `RUN_NLI = False` default | Optional GPU pass; flip to True for +recall on lexically-implicit cases |

### Things to flag in REPORT.md

- **Severe per-label imbalance.** Per-label prevalence ranges from 47.5% (`staff_positive`) down to 0.16% (`wifi_negative`, just 32 firings). NB03 must use inverse-frequency `pos_weight` per label; rare labels also need NB02b LLM augmentation.
- **`room_size_large` extractor is near-dead** (25 firings, 0.12%). Reviews rarely mention numeric room size in this corpus. The room positive/negative LFs cover comfort and condition; the extractor can be retired or kept as a reference signal.
- **LF coverage is positive-skewed**: 4 of 5 highest-coverage LFs are positive polarity. This mirrors the corpus rating distribution (74% 4-5 stars).
- **Conflict rate is informative, not a bug**: 667 reviews fire both polarities of `room` (3.3%), expected for mixed reviews like "spacious but dirty".
- **Gold set is self-annotated (n=1)**: no Cohen's kappa; treat NB04 metrics as a quality probe, not a benchmark.

### Next: Notebook 02b -- LLM Relabeling Pass

NB02b will:
1. Load the abstention pool (1,976 train-side reviews where no LF fired)
2. Format each review into a structured prompt with the 16-label taxonomy
3. Call a frontier LLM (OpenAI / Gemini / xAI -- whichever has an API
   key set) to emit JSON multi-hot labels
4. Validate outputs against a Pydantic schema with repair for common
   LLM format errors
5. Audit a 100-sample slice for quality
6. Merge into the existing silver labels to produce
   `silver_labels_augmented.parquet`

Expected uplift: most gain on rare labels (`wifi_negative`,
`breakfast_negative`, `location_negative`) since these are where
regex LFs are weakest.
